# Задание 3

In [13]:
try:
    from pyspark import SparkContext, SparkConf
    from pyspark.sql import SparkSession
except ImportError as e:
    printmd('<<<<<!!!!! Please restart your kernel after installing Apache Spark !!!!!>>>>>')

In [14]:
sc = SparkContext.getOrCreate(SparkConf().setMaster("local[*]"))

spark = SparkSession \
    .builder \
    .getOrCreate()

Все функции могут быть реализованы с использованием DataFrames, ApacheSparkSQL или RDDs. Нас интересует только результат. Вам дана ссылка на data frame в параметре "df", а если хотите использовать SQL, просто используйте параметр "spark", который является ссылкой на глобальный объект SparkSession. Наконец, если хотите использовать RDDs, просто используйте "df.rdd" для получения ссылки на базовый объект RDD. Но мы не рекомендуем использовать RDD в данный момент.

Начнем с первой функции. Рассчитайте минимальную температуру для тестового набора данных, который вы создали. Мы предоставили небольшой каркас на случай, если вы хотите использовать SQL. Все может быть реализовано только с помощью SQL, если хотите.

In [15]:
def minTemperature():
    # TODO Введите ваш код здесь, вы не обязаны использовать шаблон ниже
    # справочная информация: https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame
    return spark.sql("SELECT MIN(temperature) as mintemp FROM washing").first().mintemp

Теперь сделайте то же самое для среднего значения температуры

In [16]:
def meanTemperature():
    # TODO Введите ваш код здесь, вы не обязаны использовать шаблон ниже
    # справочная информация: https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame
    return spark.sql("SELECT AVG(temperature) as meantemp FROM washing").first().meantemp

Теперь сделайте то же самое для максимума температуры

In [17]:
def maxTemperature():
    # TODO Введите ваш код здесь, вы не обязаны использовать шаблон ниже
    # справочная информация: https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame
    return spark.sql("SELECT MAX(temperature) as maxtemp FROM washing").first().maxtemp

Теперь сделайте то же самое для стандартного отклонения температуры

In [18]:
def sdTemperature():
    # TODO Введите ваш код здесь, вы не обязаны использовать шаблон ниже
    # справочная информация: https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame
    # https://spark.apache.org/docs/2.3.0/api/sql/
    return spark.sql("SELECT STDDEV_POP(temperature) as sdtemp FROM washing").first().sdtemp

Теперь сделайте то же самое для асимметрии температуры. Поскольку SQL-выражение для этого немного сложнее, мы предоставили каркас. Вам нужно вставить пользовательский код в четырех местах, чтобы функция заработала. Альтернативно, вы можете удалить все и реализовать это самостоятельно. Обратите внимание, что мы используем две ранее определенные функции, поэтому убедитесь, что они корректны. Также обратите внимание, что мы используем возможности форматирования строк Python, где результаты двух вызовов функций "meanTemperature" и "sdTemperature" вставляются в символы "%s" в SQL-строке.

In [19]:
def skewTemperature():    
    return spark.sql("""
SELECT 
    (
        1/COUNT(*)
    ) *
    SUM (
        POWER(temperature-%s,3)/POWER(%s,3)
    )

as sktemperature from washing
                    """ %(meanTemperature(),sdTemperature())).first().sktemperature

Эксцесс - это 4-й статистический момент, так что если вы умны, вы можете использовать код для асимметрии, который является 3-м статистическим моментом. На самом деле отличаются только две вещи.

In [20]:
def kurtosisTemperature():    
        return spark.sql("""
SELECT 
    (
        1/COUNT(*)
    ) *
    SUM (
        POWER(temperature-%s,4)/POWER(%s,4)
    )
as ktemperature from washing
                    """ %(meanTemperature(),sdTemperature())).first().ktemperature

Просто подсказка. Это также можно легко решить с помощью SQL, как показано в лекции, но также и с использованием RDDs.

In [21]:
def correlationTemperatureHardness():
    # TODO Введите ваш код здесь, вы не обязаны использовать шаблон ниже
    # справочная информация: https://spark.apache.org/docs/latest/api/python/pyspark.sql.html#pyspark.sql.DataFrame
    # https://spark.apache.org/docs/2.3.0/api/sql/
    return spark.sql("SELECT CORR(temperature,hardness) as temperaturehardness FROM washing").first().temperaturehardness

Теперь пришло время взять файл PARQUET и создать из него dataframe. Используя SparkSQL, вы можете обращаться с ним как с базой данных.

In [22]:
df = spark.read.parquet('washing.parquet')
df.createOrReplaceTempView('washing')
df.show()

+--------------------+--------------------+-----+--------+----------+---------+--------+-----+-----------+-------------+-------+
|                 _id|                _rev|count|flowrate|fluidlevel|frequency|hardness|speed|temperature|           ts|voltage|
+--------------------+--------------------+-----+--------+----------+---------+--------+-----+-----------+-------------+-------+
|0d86485d0f88d1f9d...|1-57940679fb8a713...|    4|      11|acceptable|     NULL|      77| NULL|        100|1547808723923|   NULL|
|0d86485d0f88d1f9d...|1-15ff3a0b304d789...|    2|    NULL|      NULL|     NULL|    NULL| 1046|       NULL|1547808729917|   NULL|
|0d86485d0f88d1f9d...|1-97c2742b68c7b07...|    4|    NULL|      NULL|       71|    NULL| NULL|       NULL|1547808731918|    236|
|0d86485d0f88d1f9d...|1-eefb903dbe45746...|   19|      11|acceptable|     NULL|      75| NULL|         86|1547808738999|   NULL|
|0d86485d0f88d1f9d...|1-5f68b4c72813c25...|    7|    NULL|      NULL|       75|    NULL| NULL|   

Теперь давайте протестируем функции, которые вы реализовали

In [23]:
min_temperature = 0
mean_temperature = 0
max_temperature = 0
sd_temperature = 0
skew_temperature = 0
kurtosis_temperature = 0
correlation_temperature = 0

In [24]:
min_temperature = minTemperature()
print(min_temperature)

80


In [25]:
mean_temperature = meanTemperature()
print(mean_temperature)

90.03800298062593


In [26]:
max_temperature = maxTemperature()
print(max_temperature)

100


In [27]:
sd_temperature = sdTemperature()
print(sd_temperature)

6.098487624200337


In [28]:
skew_temperature = skewTemperature()
print(skew_temperature)

0.006788255973582835


In [29]:
kurtosis_temperature = kurtosisTemperature()
print(kurtosis_temperature)

1.158158434967638


In [30]:
correlation_temperature = correlationTemperatureHardness()
print(correlation_temperature)

0.017754069047296324


Поздравляем, вы закончили, отправьте этот ноутбук на проверку.